In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [4]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")  # 프리필링
    text = chat(messages, stop_sequences=["```"])  # 정지 시퀀스
    return json.loads(text)

In [5]:
dataset = generate_dataset()

dataset

[{'task': "Write a Python function that extracts the AWS account ID from an ARN (Amazon Resource Name) string. For example, given 'arn:aws:s3:::my-bucket/my-object', it should handle cases where the account ID may or may not be present."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'company-data'. The policy should include appropriate principals, actions, resources, and effect settings."},
 {'task': "Write a regular expression pattern that matches valid AWS EC2 instance IDs. Instance IDs follow the format 'i-' followed by 8 or 17 hexadecimal characters (e.g., i-1234567890abcdef0)."}]

In [6]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)